# SegRNN — Colab runner
Runtime → Change runtime type → **T4 GPU** before running.
Dataset CSVs must already be in Google Drive at `MyDrive/ts-project/dataset/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

In [ ]:
# Short run to verify the pipeline end to end. NOT a result.
# Flags reconciled against scripts/SegRNN/etth1.sh (the source of truth for paper hyperparameters)
# and run_longExp.py's argparse: added the required --model_id, switched --seg_len 48 -> 24,
# and added --enc_in/--dropout/--rnn_type/--dec_way/--channel_id/--batch_size/--learning_rate
# to match the reference script (train_epochs/itr are overridden here since this is only a smoke test).
!python -u run_longExp.py \
  --is_training 1 --model_id ETTh1_720_96 --model SegRNN --data ETTh1 \
  --root_path ./dataset/ --data_path ETTh1.csv \
  --features M --seq_len 720 --pred_len 96 --seg_len 24 --enc_in 7 \
  --d_model 512 --dropout 0.1 --rnn_type gru --dec_way pmf --channel_id 1 \
  --batch_size 64 --learning_rate 0.0003 \
  --train_epochs 2 --itr 1

In [ ]:
!sh scripts/SegRNN/etth1.sh

In [ ]:
# Classical baselines (naive, seasonal naive) + MASE, on the exact same
# data pipeline/split/scaling as the SegRNN runs above -- see scripts/baselines.py
# and docs/data_pipeline_audit.md. Deterministic, no GPU needed, seconds to run.
for pred_len in [96, 192, 336, 720]:
    !python scripts/baselines.py --data ETTh1 --root_path ./dataset/ \
      --data_path ETTh1.csv --seq_len 720 --pred_len {pred_len}